In [23]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg


# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 데이터 로드

In [24]:
df_c = pd.read_csv("../../tft_game_dataset/TFT_Challenger_MatchData.csv")
df_g = pd.read_csv("../../tft_game_dataset/TFT_GrandMaster_MatchData.csv")
df_m = pd.read_csv("../../tft_game_dataset/TFT_Master_MatchData.csv")
df_d = pd.read_csv("../../tft_game_dataset/TFT_Diamond_MatchData.csv")
df_p = pd.read_csv("../../tft_game_dataset/TFT_Platinum_MatchData.csv")

df_champion = pd.read_csv("../../tft_game_dataset/TFT_Champion_CurrentVersion.csv")
df_item = pd.read_csv("../../tft_game_dataset/TFT_Item_CurrentVersion.csv")

### 각 테이블에 티어 컬럼 추가

In [25]:
df_c["tier"] = "Challenger"
df_g["tier"] = "GrandMaster"
df_m["tier"] = "Master"
df_d["tier"] = "Diamond"
df_p["tier"] = "Platinum"

## 매치데이터 머지

In [26]:
merged_df = pd.concat([df_c, df_g, df_m, df_d, df_p], ignore_index=True)

In [27]:
print(merged_df.shape)
print(merged_df.columns)
print(merged_df.info())

(399998, 9)
Index(['gameId', 'gameDuration', 'level', 'lastRound', 'Ranked',
       'ingameDuration', 'combination', 'champion', 'tier'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 399998 entries, 0 to 399997
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   gameId          399998 non-null  str    
 1   gameDuration    399998 non-null  float64
 2   level           399998 non-null  int64  
 3   lastRound       399998 non-null  int64  
 4   Ranked          399998 non-null  int64  
 5   ingameDuration  399998 non-null  float64
 6   combination     399998 non-null  str    
 7   champion        399998 non-null  str    
 8   tier            399998 non-null  str    
dtypes: float64(2), int64(3), str(4)
memory usage: 27.5 MB
None


In [28]:
merged_df.head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4247538593,2142.470703,8,35,1,2134.272217,"{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'S...","{'JarvanIV': {'items': [27], 'star': 3}, 'Sona...",Challenger
1,KR_4247538593,2142.470703,9,35,2,2134.272217,"{'Blaster': 2, 'Mercenary': 1, 'Rebel': 6, 'Se...","{'Malphite': {'items': [7], 'star': 2}, 'Yasuo...",Challenger
2,KR_4247538593,2142.470703,8,34,3,2073.459229,"{'Cybernetic': 1, 'DarkStar': 3, 'Demolitionis...","{'KaiSa': {'items': [99, 2, 23], 'star': 2}, '...",Challenger
3,KR_4247538593,2142.470703,8,33,4,1998.146729,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 1,...","{'KaiSa': {'items': [44, 37], 'star': 2}, 'Ann...",Challenger
4,KR_4247538593,2142.470703,9,33,5,1986.443237,"{'Blaster': 2, 'Demolitionist': 2, 'Mercenary'...","{'Ziggs': {'items': [], 'star': 1}, 'Yasuo': {...",Challenger


In [35]:
# 중복행
duplicates = merged_df[merged_df.duplicated(keep=False)]

print(f"중복 제거 전: {len(merged_df)}")
print(f"중복된 행 개수: {len(duplicates)}")
print()

# 중복행 샘플
print(duplicates.head(20))
print()

# 중복 제거
merged_df = merged_df.drop_duplicates()
print(f"중복 제거 후: {len(merged_df)}")

중복 제거 전: 399998
중복된 행 개수: 80

               gameId  gameDuration  level  lastRound  Ranked  ingameDuration  \
100911  KR_4336328756    125.278297      3          4       0      125.278297   
100912  KR_4336328756    125.278297      3          4       0      125.278297   
100913  KR_4336328756    125.278297      3          4       0      125.278297   
100914  KR_4336328756    125.278297      3          4       0      125.278297   
100915  KR_4336328756    125.278297      3          4       0      125.278297   
100916  KR_4336328756    125.278297      3          4       0      125.278297   
100917  KR_4336328756    125.278297      3          4       0      125.278297   
166703  KR_4295022760   1714.133789      4         10       0     1714.133789   
166704  KR_4295022760   1714.133789      4         10       0     1714.133789   
166705  KR_4295022760   1714.133789      4         10       0     1714.133789   
166706  KR_4295022760   1714.133789      4         10       0     1714.133789  

## combination 컬럼

combination 컬럼이 {} 이거나 champion 컬럼이 {} 이거나

In [36]:
problem_rows = merged_df[
    (merged_df['combination'].astype(str).str.strip() == '{}') |
    (merged_df['champion'].astype(str).str.strip() == '{}')
]

problem_rows[['gameId', 'combination', 'champion', 'tier']]

,gameId,combination,champion,tier
345,KR_4265026918,{},{},Challenger
1719,KR_4274400668,{},{},Challenger
2426,KR_4280509972,{},"{'Fiora': {'items': [], 'star': 1}, 'Leona': {...",Challenger
2521,KR_4280775299,{},"{'KaiSa': {'items': [], 'star': 2}, 'Annie': {...",Challenger
2839,KR_4282545658,{},"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': ...",Challenger
...,...,...,...,...
387957,KR_4356646200,{},{},Platinum
388329,KR_4314848881,{},{},Platinum
389000,KR_4327374555,{},{},Platinum
391468,KR_4294692860,{},"{'TwistedFate': {'items': [], 'star': 2}, 'Mal...",Platinum


In [37]:
comb_empty = merged_df['combination'].astype(str).str.strip() == '{}'
champ_empty = merged_df['champion'].astype(str).str.strip() == '{}'

print("combination 컬럼이 {} 인 행 개수:", comb_empty.sum())
print("champion 컬럼이 {} 인 행 개수:", champ_empty.sum())
print("두 개 다 {} 인 행 개수:", (comb_empty & champ_empty).sum())
print("champion은 있는데 combination만 {} 인 행 개수:", (comb_empty & ~champ_empty).sum())

combination 컬럼이 {} 인 행 개수: 361
champion 컬럼이 {} 인 행 개수: 333
두 개 다 {} 인 행 개수: 252
champion은 있는데 combination만 {} 인 행 개수: 109


In [38]:
#gameId별 개수
print("combination이 {} 인 gameId 개수:", merged_df.loc[comb_empty, 'gameId'].nunique())
print("champion이 {} 인 gameId 개수:", merged_df.loc[champ_empty, 'gameId'].nunique())
print("둘 다 {} 인 gameId 개수:", merged_df.loc[comb_empty & champ_empty, 'gameId'].nunique())
print("champion은 있는데 combination만 {} 인 gameId 개수:", merged_df.loc[comb_empty & ~champ_empty, 'gameId'].nunique())

combination이 {} 인 gameId 개수: 313
champion이 {} 인 gameId 개수: 240
둘 다 {} 인 gameId 개수: 206
champion은 있는데 combination만 {} 인 gameId 개수: 109


In [39]:
merged_df.loc[comb_empty & ~champ_empty, ['gameId', 'combination', 'champion', 'tier']].sort_values('gameId')

,gameId,combination,champion,tier
326625,KR_4271274609,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {...",Platinum
162174,KR_4275570139,{},"{'Fiora': {'items': [], 'star': 1}, 'Leona': {...",Master
2426,KR_4280509972,{},"{'Fiora': {'items': [], 'star': 1}, 'Leona': {...",Challenger
2521,KR_4280775299,{},"{'KaiSa': {'items': [], 'star': 2}, 'Annie': {...",Challenger
2839,KR_4282545658,{},"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': ...",Challenger
...,...,...,...,...
261493,KR_4380809484,{},"{'Ziggs': {'items': [], 'star': 2}, 'Malphite'...",Diamond
301467,KR_4383383173,{},"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': ...",Diamond
295376,KR_4392159078,{},"{'Malphite': {'items': [], 'star': 2}, 'Caitly...",Diamond
259609,KR_4393798544,{},"{'JarvanIV': {'items': [], 'star': 2}, 'Xayah'...",Diamond
